# Explorer les graphes en Python

Ce carnet suit les quatre familles de l’application en cinq études : villes, dépendances temporelles, machines cycliques, DAG de production et concordances nodales. Les données sont fictives ; les résultats sont **calculés**, jamais présentés comme des mesures.

Exécuter les cellules dans l’ordre. Les fonctions et les figures proviennent des mêmes modules Python que l’application Streamlit. Aucun serveur Symfony, Docker ou moteur JavaScript n’est nécessaire. Les sorties sont volontairement absentes du fichier distribué.

In [ ]:
from pathlib import Path
import sys

# Deux répertoires de travail acceptés : python/ ou python/notebooks/.
cwd = Path.cwd().resolve()
if (cwd / "physique_graphes").is_dir():
    PYTHON_ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "physique_graphes").is_dir():
    PYTHON_ROOT = cwd.parent
else:
    raise RuntimeError("Lancer ce carnet depuis python/ ou python/notebooks/.")
if str(PYTHON_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_ROOT))

from copy import deepcopy
import math
import numpy as np
import pandas as pd
from IPython.display import display
from physique_graphes import reseaux as r, production as p, dag, concordances as c
from physique_graphes.optimisation import optimiser
from physique_graphes.visualisation import graphe, courbes, echantillonner_surface, nappe
from examples.lois_personnelles import PERSONAL_LAWS

print("Modules locaux chargés ; modèles modifiables dans les cellules suivantes.")

## 1. Villes : positions, distances et chemins

Le modèle initial contient six villes et neuf liaisons bidirectionnelles. Les coordonnées sont en kilomètres. On fixe le départ **et** l’arrivée, puis on calcule la distance et le chemin. Les trois algorithmes départagent les égalités par le nombre d’arcs, puis par les identifiants.

In [ ]:
villes = r.villes_exemple()                 # 9 liaisons, coordonnées modifiables
pondere = r.ponderer_villes(villes)          # 18 arcs dirigés, poids euclidiens
resultats = {
    "Dijkstra": r.dijkstra(pondere, "A", "F"),
    "Bellman–Ford": r.bellman_ford(pondere, "A", "F"),
    "Floyd–Warshall": r.floyd_warshall(pondere, "A", "F"),
}
assert all(v["distance"] == 12 for v in resultats.values())
display(pd.DataFrame.from_dict(resultats, orient="index"))
display(graphe(villes, directed=False, title="Six villes fictives · distances en km"))

In [ ]:
parcours = r.parcours_bornes(
    pondere, 18, depart="A", arrivee="F", max_results=5000, max_expansions=100000
)
print("Liste complète :", parcours["complete"], "—", len(parcours["paths"]), "chemins")
if parcours["warning"]:
    print(parcours["warning"])
display(pd.DataFrame(parcours["paths"]))
assert all(row["distance"] < 18 for row in parcours["paths"])
# Aucun sommet répété ; pas de trajet avec cycles ni de trajet vide.

## 2. Dépendance à un débit externe

Avec un débit fixé `q` en **véhicules/min**, les durées sont `T_ABD=4+q/10` et `T_ACD=8` minutes. La meilleure durée est continue, mais sa dérivée change à `q=40`. Il s’agit d’une comparaison statique, pas d’un calcul de trafic endogène.

In [ ]:
debits = list(range(101))
etudes = [r.routes_dependantes(q) for q in debits]
display(courbes(debits, {
    "A–B–D": [row["routes"][0]["duration"] for row in etudes],
    "A–C–D": [row["routes"][1]["duration"] for row in etudes],
    "Meilleure durée": [row["bestDuration"] for row in etudes],
}, xlabel="Débit externe (véhicules/min)", ylabel="Durée (min)"))
seuil = r.sensibilite_dependance(40)
print("Au seuil :", [route["id"] for route in seuil["best"]])
print("Dérivées à gauche / droite :", seuil["leftDerivative"], seuil["rightDerivative"], seuil["derivativeUnit"])
assert seuil["bestDerivative"] is None  # Ne pas inventer une dérivée au point anguleux.

## 3. Machines couplées : cycles synchrones

Les sorties `y[k]` sont lues simultanément pour calculer l’état suivant. L’alimentation est plafonnée à 1, avec surplus explicite non stocké. Le résidu est recalculé **après** le dernier cycle : il teste le point fixe, pas un optimum.

Le scénario sous seuil dépend de l’initialisation. L’alternance observée sur un horizon fini ne démontre pas la stabilité d’un cycle asymptotique.

In [ ]:
simulations = {name: p.simuler_production(p.production_exemple(name), cycles=40)
               for name in ("balanced", "threshold", "oscillating")}
display(pd.DataFrame([
    {"scénario": name, "sorties finales": row["finalOutputs"], "résidu": row["residual"],
     "statut": row["status"], "alternance observée": row["period2Observed"]}
    for name, row in simulations.items()
]))
assert np.allclose(simulations["balanced"]["finalOutputs"], [.275, .265, .1875], atol=1e-10)
assert simulations["threshold"]["finalOutputs"] == [0, 0, 0]
assert simulations["oscillating"]["period2Observed"]
display(courbes(list(range(41)), {
    "M1 · équilibré": [row["outputs"][0] for row in simulations["balanced"]["history"]],
    "M1 · oscillant": [row["outputs"][0] for row in simulations["oscillating"]["history"]],
}, xlabel="Cycle synchrone", ylabel="Production normalisée"))

In [ ]:
amorce = p.production_exemple("threshold")
for machine in amorce["machines"]:
    machine["initial"] = .8
non_nul = p.simuler_production(amorce)
print("Départ .8 :", non_nul["finalOutputs"], "; point fixe homogène 27/35 =", 27/35)
assert np.allclose(non_nul["finalOutputs"], [27/35] * 3)
# Le même modèle admet aussi l’état homogène .4 ; zéro n’est pas son unique équilibre.
assert np.allclose(p.pas_production(amorce, [.4] * 3)["outputs"], [.4] * 3)

## 4. DAG statique : douze branches actives et loi personnelle

Ici les transformations portent sur les **branches**, avec sortie `g(x)=x f(x)`. Les nœuds somment les apports ; il n’y a ni cycle temporel, ni écrêtage, ni stock. Le modèle initial reprend le préréglage actif du site, de production `143217/320000 = 0,447553125`.

La recherche SciPy ci-dessous est locale et ne délivre pas de certificat global. Une fois une loi modifiée, la preuve particulière du préréglage ne s’applique plus automatiquement.

In [ ]:
modele_dag = dag.create_branch_active()
etat_dag = dag.evaluate(modele_dag)
assert etat_dag["feasible"]
assert math.isclose(etat_dag["production"], 143217/320000, abs_tol=1e-14)
assert all(row["input"] > 0 and row["output"] > 0 for row in etat_dag["edges"].values())
print("Production :", etat_dag["production"], "; bilan global :", etat_dag["balance_residual"])
display(pd.DataFrame(etat_dag["edges"]).T)
display(graphe(modele_dag, {n: row["output"] for n, row in etat_dag["nodes"].items()},
               etat_dag["y"], title="DAG : lois sur les branches"))

In [ ]:
parts_dag = dag.binary_controls(modele_dag)
def production_dag(s1, s2):
    candidat = dag.with_binary_controls(modele_dag, {**parts_dag, "1": s1, "2": s2})
    etat = dag.evaluate(candidat)
    return etat["production"] if etat["feasible"] else None

xs, ys, zs = echantillonner_surface(production_dag, parts_dag["1"], parts_dag["2"],
                                  radius=.12, points=21, bounds=[(0, 1), (0, 1)])
figure_dag = nappe(xs, ys, zs, xlabel="Part s1", ylabel="Part s2", zlabel="Production compatible",
                  reference=(parts_dag["1"], parts_dag["2"], etat_dag["production"]))
display(figure_dag)
recherche_dag = optimiser(modele_dag, method="local", iterations=25)
print({key: value for key, value in recherche_dag.items() if key != "best"})
assert recherche_dag["certified"] is False and recherche_dag["upper_bound"] is None

In [ ]:
# Le registre vient de examples/lois_personnelles.py ; le JSON ne contient pas de code.
personnalise = deepcopy(modele_dag)
personnalise["edges"][0]["law"] = {"name": "rendement_constant", "parameters": {"eta": .9}}
personnalise["edges"][0]["attributes"]["commentaire"] = "Loi personnelle : sortie = .9 × entrée."
etat_personnel = dag.evaluate(personnalise, registry=PERSONAL_LAWS)
assert etat_personnel["feasible"]
print("Production après modification :", etat_personnel["production"])
# Une fonction du registre renvoie directement la production, pas seulement f(x).
# Facultatif, pour conserver votre modèle :
# dag.save_model(personnalise, PYTHON_ROOT / "mon-modele.json")

## 5. Concordances : graine 34, mode signé, dérivées et lagrangien

La transformation est désormais **nodale** : `X_i=Σq_ji`, `C_i=e_i+Σε_ij q_ji`, puis `Y_i=X_i C_i` en signé. Les sorties négatives restent négatives et sont réparties avec leur signe. En rectifié, la loi devient `max(0,X_i C_i)`. Aucun plafond positif n’est ajouté : `Y8≤1` n’est pas une règle de ce modèle.

La graine 34 tire 56 coefficients hors diagonale ; seuls les douze arcs du DAG contribuent. Les environnements sont ici tous égaux à 0,5. Une comparaison en un point commun ne prouve pas une relation entre les deux maxima globaux.

In [ ]:
base = c.create_scenario()
base["environments"] = {str(i): .5 for i in range(2, 9)}
modele_c = c.randomize(base, seed=34)
initial_signe = c.evaluate(modele_c, domain="signed")
initial_rectifie = c.evaluate(modele_c, domain="rectified")
display(pd.DataFrame([
    {"mode": name, "Y8": state["objective"], "X8": state["objectives"]["algebraic_arrivals"],
     "somme absolue arrivées": state["objectives"]["arrivals"]}
    for name, state in (("signé", initial_signe), ("rectifié", initial_rectifie))
]))
assert initial_signe["derivatives"]["differentiable"]
cles = c.GRAPH["controls"]
display(pd.DataFrame({"partage": cles, "dY8/ds au départ": initial_signe["derivatives"]["gradient"]}))
display(pd.DataFrame(initial_signe["derivatives"]["hessian"], index=cles, columns=cles))

# Vérification locale indépendante des cinq dérivées au départ intérieur (.5,...,.5).
h = 1e-6
finies = []
for key in cles:
    plus = {**modele_c["initial_controls"], key: .5+h}
    moins = {**modele_c["initial_controls"], key: .5-h}
    finies.append((c.evaluate(modele_c, plus, domain="signed", detailed=False)["objective"]
                   - c.evaluate(modele_c, moins, domain="signed", detailed=False)["objective"]) / (2*h))
assert np.allclose(finies, initial_signe["derivatives"]["gradient"], rtol=1e-6, atol=1e-7)
print("Écart maximal des différences finies :", max(abs(a-b) for a,b in zip(finies, initial_signe["derivatives"]["gradient"])))

In [ ]:
# Budgets explicites : ne pas assimiler meilleur témoin et optimum certifié.
locale = c.search_local(modele_c, domain="signed", max_evaluations=500)
globale = c.search_global(modele_c, domain="signed", max_nodes=80, tolerance=1e-5)
display(pd.DataFrame([
    {"méthode": row["method"], "statut": row["status"], "meilleur Y8": row["best"]["objective"] if row["best"] else None,
     "borne supérieure": row["upper_bound"], "écart": row["gap"], "évaluations": row["evaluations"]}
    for row in (locale, globale)
]))
etat_c = locale["best"] or initial_signe
parts_c = etat_c["controls"]
print("Référence retenue pour les figures : meilleur témoin local, sans preuve de maximum global.")
display(pd.DataFrame(etat_c["nodes"]))

In [ ]:
def production_concordances(s1, s2):
    return c.evaluate(modele_c, {**parts_c, "s1": s1, "s2": s2},
                      domain="signed", detailed=False)["objective"]
xs, ys, zs = echantillonner_surface(production_concordances, parts_c["s1"], parts_c["s2"],
                                  radius=.15, points=21, bounds=[(0, 1), (0, 1)])
figure_compatible = nappe(xs, ys, zs, xlabel="s1", ylabel="s2", zlabel="Y8 compatible",
                         reference=(parts_c["s1"], parts_c["s2"], etat_c["objective"]))
display(figure_compatible)

### Une coupe libre de L n’est pas une production réalisable

Le lagrangien utilise 26 coordonnées libres : les douze transferts, `X2…X8` et `Y2…Y8`. Ses 21 égalités sont les sept définitions d’entrée, les sept lois nodales et les sept bilans sortants (aucun bilan sortant en 8).

`L = Y8 + Σ λ_i(X_i−Σq_ji) + Σ μ_i(Y_i−X_i C_i) + Σ η_i(Σq_ik−Y_i)` en signé.

Les multiplicateurs adjoints sont calculés à la référence puis **fixés**. La coupe ci-dessous varie `q2→8` et `X8`, en laissant les 24 autres coordonnées fixes : elle ne réapplique pas les contraintes. Une selle ou une pente ne sont pas remplacées par un maximum fictif. La nullité de dérivées ne suffirait pas à prouver un optimum global.

In [ ]:
explication = c.explain_lagrangian(modele_c, etat_c)
point = explication["reference_point"]
multiplicateurs = explication["multipliers"]
reference_l = explication["at_reference"]
assert len(point) == 26 and len(multiplicateurs) == 21
assert reference_l["residual"] < 1e-10
assert math.isclose(reference_l["lagrangian"], etat_c["objective"], abs_tol=1e-9)

noms = ["q" + edge["id"] for edge in c.GRAPH["edges"]] + [f"X{i}" for i in range(2, 9)] + [f"Y{i}" for i in range(2, 9)]
ix, iy = noms.index("q2-8"), noms.index("X8")
def lagrangien_libre(q28, x8):
    libre = list(point)
    libre[ix], libre[iy] = q28, x8
    return c.evaluate_lagrangian(modele_c, libre, multipliers=multiplicateurs,
                                 domain="signed")["lagrangian"]
xs, ys, zs = echantillonner_surface(lagrangien_libre, point[ix], point[iy], radius=.10, points=21)
figure_libre = nappe(xs, ys, zs, xlabel="q2→8 libre", ylabel="X8 libre", zlabel="L hors contraintes",
                    reference=(point[ix], point[iy], reference_l["lagrangian"]))
display(figure_libre)
print("Dérivées libres de L à la référence :", {noms[i]: reference_l["gradient"][i] for i in (ix, iy)})
print("Diagnostic adjoint = certificat global ?", explication["global_certificate"])
assert explication["global_certificate"] is False

## Conserver une étude et poursuivre

Les objets `modele_dag`, `personnalise` et `modele_c` sont des dictionnaires modifiables. Les figures Plotly sont réutilisables dans Streamlit ou exportables en HTML autonome. Les lignes suivantes sont commentées pour que **Exécuter toutes les cellules** ne crée aucun fichier supplémentaire.

Les solveurs globaux PL/spatiaux du site pour les machines et branches, ainsi que leurs lagrangiens particuliers à 24 variables, ne sont pas portés ici. Les recherches SciPy des DAG restent heuristiques ; les bornes de concordances gardent leur statut et leur budget.

In [ ]:
# Exports facultatifs : décommenter après avoir choisi les noms souhaités.
# dag.save_model(personnalise, PYTHON_ROOT / "mon-dag.json")
# figure_compatible.write_html(PYTHON_ROOT / "ma-nappe-compatible.html", include_plotlyjs=True)
# figure_libre.write_html(PYTHON_ROOT / "ma-coupe-libre.html", include_plotlyjs=True)
print("Parcours terminé : états, dérivées, témoins et bornes restent distingués.")